# Round 4 — kickoff

Companion notes for the website log in `imc_logs/ROUND_4/`, the **wait-on-tape** idea (`ROUND_4/trader_WAITALPHA.py`), named bot-flow overlays (`ROUND_4/trader_BOTVELVET.py` / `ROUND_4/trader_HYPER.py`), and `own_trades` vs `market_trades`.

In [ ]:
from __future__ import annotations
import json
import io
import csv
from collections import Counter
from pathlib import Path

ROOT = Path("../..").resolve()
LOG = ROOT / "imc_logs" / "ROUND_4" / "503837.log"
data = json.loads(LOG.read_text())
th = data["tradeHistory"]
print("keys:", list(data.keys()))
print("tradeHistory len:", len(th))

In [ ]:
# Who traded with us / each other? "SUBMISSION" = you on the IMC site.
buyers, sellers = Counter(), Counter()
for t in th:
    buyers[t["buyer"]] += 1
    sellers[t["seller"]] += 1
print("Top buyers:", buyers.most_common(8))
print("Top sellers:", sellers.most_common(8))
bb = [t for t in th if t["buyer"] != "SUBMISSION" and t["seller"] != "SUBMISSION"]
print("Trades with no SUBMISSION (bot↔bot prints):", len(bb))

In [ ]:
act = data["activitiesLog"]
rows = list(csv.DictReader(io.StringIO(act), delimiter=";"))
days = {r["day"] for r in rows}
print("days in file:", sorted(days, key=int))
# last timestamp rows — per-product pnl (mark-to-market)
last_ts = max(int(r["timestamp"]) for r in rows)
r_last = [r for r in rows if r["day"] == max(days, key=int) and int(r["timestamp"]) == last_ts]
print("total final pnl:", sum(float(r["profit_and_loss"]) for r in r_last))
for r in sorted(r_last, key=lambda x: x["product"])[:6]:
    print(r["product"], r["profit_and_loss"])

## `own_trades` vs `market_trades` (in your `run(state)`)

Both are `dict[product, list[Trade]]` on `TradingState` (see `datamodel.py`).

- **`own_trades`**: only prints where you are **buyer** or **seller** (in the backtest, the id is `"SUBMISSION"`). Use this for *your* PnL and inventory flow.
- **`market_trades`**: the **public tape** for the slice the engine exposes. In the local backtester, any public tape quantity that fills your order is removed from `market_trades` and appears in `own_trades` instead.

So for our local tools: **treat them as disjoint, not subset/superset**. To count all prints that affected the tick, union `own_trades + market_trades`. To build bot signals without self-contamination, use `market_trades` only or explicitly filter `SUBMISSION` out of the union.

On the IMC website this is worth testing directly, which is why `_trader_BOTNAME_PROBE.py` logs both channels separately.

In [ ]:
# Reproduce tooling (from repo root)
print("uv run rank --show-per-product")
print("uv run rank --show-per-product --trader trader_BOTVELVET.py")
print("uv run rank --show-per-product --trader trader_HYPER.py")
print("uv run check_overfit --all --by-pnl")
print("uv run rank --trader _trader_BOTNAME_PROBE.py   # bot probe, not in default discovery")